# Create FAPEMIG Awards from Official Open Data

Creates OpenAlex award rows from FAPEMIG's official `PROJETOS CONTRATADOS` open-data workbook.

- Funder: Fundação de Amparo à Pesquisa do Estado de Minas Gerais (`F4320322980`; ROR `https://ror.org/00nc55f03`; DOI `10.13039/501100004901`)
- Source authority: FAPEMIG's own Dados Abertos page and linked Projetos Contratados XLSX.
- Source method: official source ladder 4, bulk workbook download from first-party open data.
- Local validation on 2026-05-27: 23,586 unique process numbers, year range 2007-2025, 99.9% amount/date coverage, BRL 4.12B total.
- Provenance: `fapemig_projetos_contratados`; priority: `142`.


## Step 1: Create Raw Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fapemig_projetos_contratados_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/fapemig/fapemig_projetos_contratados.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 23,586 rows.
SELECT COUNT(*) AS total_rows
FROM openalex.awards.fapemig_projetos_contratados_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.fapemig_projetos_contratados_raw;


In [ ]:
%sql
SELECT
    funder_award_id,
    processo,
    display_name,
    coordinator,
    modalidade,
    amount,
    currency,
    source_year,
    start_date,
    instituicaoexecutora_nome
FROM openalex.awards.fapemig_projetos_contratados_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys


In [ ]:
%sql
-- Money/date-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'fapemig_projetos_contratados_raw'
  AND LOWER(column_name) RLIKE 'amount|valor|currency|year|date|data|total|fund|grant|award|processo'
ORDER BY column_name;


In [ ]:
%sql
-- Coverage profile from the official workbook.
SELECT
    COUNT(*) AS total_rows,
    COUNT(processo) AS processo_rows,
    ROUND(try_divide(COUNT(processo), COUNT(*)) * 100.0, 1) AS pct_processo,
    COUNT(display_name) AS display_name_rows,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_display_name,
    COUNT(source_title) AS source_title_rows,
    ROUND(try_divide(COUNT(source_title), COUNT(*)) * 100.0, 1) AS pct_source_title,
    COUNT(coordinator) AS coordinator_rows,
    ROUND(try_divide(COUNT(coordinator), COUNT(*)) * 100.0, 1) AS pct_coordinator,
    COUNT(amount) AS amount_rows,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(currency) AS currency_rows,
    ROUND(try_divide(COUNT(currency), COUNT(*)) * 100.0, 1) AS pct_currency,
    COUNT(start_date) AS start_date_rows,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_start_date,
    COUNT(end_date) AS end_date_rows,
    ROUND(try_divide(COUNT(end_date), COUNT(*)) * 100.0, 1) AS pct_end_date,
    COUNT(instituicaoexecutora_nome) AS executing_institution_rows,
    ROUND(try_divide(COUNT(instituicaoexecutora_nome), COUNT(*)) * 100.0, 1) AS pct_executing_institution
FROM openalex.awards.fapemig_projetos_contratados_raw;


In [ ]:
%sql
-- Native-key inspection. processo is unique and backs funder_award_id.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT processo) AS distinct_processos,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids
FROM openalex.awards.fapemig_projetos_contratados_raw;


In [ ]:
%sql
-- Must return zero rows.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.fapemig_projetos_contratados_raw
GROUP BY funder_award_id
HAVING COUNT(*) > 1
ORDER BY rows DESC, funder_award_id;


In [ ]:
%sql
-- Scheme and funding-type distribution.
SELECT
    funding_type,
    modalidade,
    COUNT(*) AS rows,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_brl
FROM openalex.awards.fapemig_projetos_contratados_raw
GROUP BY funding_type, modalidade
ORDER BY rows DESC
LIMIT 50;


In [ ]:
%sql
-- Year and amount range.
SELECT
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year,
    ROUND(MIN(TRY_CAST(amount AS DOUBLE)), 2) AS min_amount_brl,
    ROUND(MAX(TRY_CAST(amount AS DOUBLE)), 2) AS max_amount_brl,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_amount_brl
FROM openalex.awards.fapemig_projetos_contratados_raw;


## Step 1.6: Funder Existence Fail-Fast


In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for FAPEMIG F4320322980'
) AS funder_exists
FROM openalex.common.funder
WHERE funder_id = 4320322980;


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320322980;


## Step 2: Transform to OpenAlex Award Schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fapemig_awards
USING delta
AS
WITH
fapemig_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320322980
),
raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        NULLIF(TRIM(currency), '') AS parsed_currency,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_year
    FROM openalex.awards.fapemig_projetos_contratados_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS id,
        TRIM(g.display_name) AS display_name,
        CASE WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL ELSE TRIM(g.description) END AS description,
        f.funder_id,
        g.native_award_id AS funder_award_id,
        g.parsed_amount AS amount,
        g.parsed_currency AS currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        COALESCE(NULLIF(TRIM(g.funding_type), ''), 'grant') AS funding_type,
        COALESCE(NULLIF(TRIM(g.funder_scheme), ''), NULLIF(TRIM(g.modalidade), ''), 'Projetos contratados') AS funder_scheme,
        'fapemig_projetos_contratados' AS provenance,
        g.parsed_start_date AS start_date,
        g.parsed_end_date AS end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_year) AS start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_year) AS end_year,
        CASE
            WHEN g.coordinator IS NULL OR TRIM(g.coordinator) = '' THEN CAST(NULL AS STRUCT<
                given_name:STRING,
                family_name:STRING,
                orcid:STRING,
                role_start:DATE,
                affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
            >)
            ELSE struct(
                NULLIF(TRIM(g.coordinator_given_name), '') AS given_name,
                NULLIF(TRIM(g.coordinator_family_name), '') AS family_name,
                CAST(NULL AS STRING) AS orcid,
                g.parsed_start_date AS role_start,
                struct(
                    NULLIF(TRIM(g.instituicaoexecutora_nome), '') AS name,
                    'BR' AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        END AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        NULLIF(TRIM(g.landing_page_url), '') AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) AS works_api_url,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM raw_prepared g
    CROSS JOIN fapemig_funder f
)
SELECT * FROM awards_transformed;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_award_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_native_ids,
    ROUND(SUM(amount), 2) AS total_amount_brl
FROM openalex.awards.fapemig_awards;


In [ ]:
%sql
SELECT
    id,
    display_name,
    funder_award_id,
    amount,
    currency,
    funding_type,
    funder_scheme,
    start_year,
    lead_investigator.family_name AS lead_family_name,
    lead_investigator.affiliation.name AS affiliation
FROM openalex.awards.fapemig_awards
LIMIT 20;


## Step 3: Delete Existing Rows and Insert Fresh Priority 142 Rows


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fapemig_projetos_contratados' AND priority = 142;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    142 AS priority
FROM openalex.awards.fapemig_awards;


In [ ]:
%sql
SELECT
    provenance,
    priority,
    COUNT(*) AS rows,
    COUNT(DISTINCT funder_award_id) AS distinct_native_ids,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    ROUND(SUM(amount), 2) AS total_amount_brl
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fapemig_projetos_contratados'
GROUP BY provenance, priority;


## Step 6: Verification Queries


In [ ]:
%sql
-- Duplicate OpenAlex IDs within this ingest: must return zero rows.
SELECT id, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fapemig_projetos_contratados' AND priority = 142
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY rows DESC, id;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS title_rows,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    COUNT(funder_award_id) AS native_id_rows,
    ROUND(try_divide(COUNT(funder_award_id), COUNT(*)) * 100.0, 1) AS pct_native_id,
    COUNT(amount) AS amount_rows,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(currency) AS currency_rows,
    ROUND(try_divide(COUNT(currency), COUNT(*)) * 100.0, 1) AS pct_currency,
    COUNT(start_year) AS start_year_rows,
    ROUND(try_divide(COUNT(start_year), COUNT(*)) * 100.0, 1) AS pct_start_year,
    COUNT(lead_investigator.family_name) AS coordinator_rows,
    ROUND(try_divide(COUNT(lead_investigator.family_name), COUNT(*)) * 100.0, 1) AS pct_coordinator,
    COUNT(lead_investigator.affiliation.name) AS affiliation_rows,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name), COUNT(*)) * 100.0, 1) AS pct_affiliation
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fapemig_projetos_contratados' AND priority = 142;


In [ ]:
%sql
SELECT
    funding_type,
    funder_scheme,
    COUNT(*) AS rows,
    COUNT(amount) AS amount_rows,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(SUM(amount), 2) AS total_amount_brl
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fapemig_projetos_contratados' AND priority = 142
GROUP BY funding_type, funder_scheme
ORDER BY rows DESC
LIMIT 50;


In [ ]:
%sql
SELECT
    funder_id,
    funder.display_name AS funder_name,
    provenance,
    priority,
    COUNT(*) AS rows,
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year,
    ROUND(SUM(amount), 2) AS total_amount_brl
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fapemig_projetos_contratados' AND priority = 142
GROUP BY funder_id, funder.display_name, provenance, priority;


In [ ]:
%sql
SELECT
    id,
    display_name,
    description,
    funder_award_id,
    amount,
    currency,
    funding_type,
    funder_scheme,
    start_date,
    end_date,
    lead_investigator,
    landing_page_url,
    works_api_url
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fapemig_projetos_contratados' AND priority = 142
ORDER BY funder_award_id
LIMIT 25;


## Handoff / Admin Notes

Contractor-complete handoff only. The script and notebook are ready, but the contractor has no S3 or Databricks access.

Admin must:

- Run `scripts/local/fapemig_to_s3.py` without `--skip-upload` to upload `fapemig_projetos_contratados.parquet`.
- Run this Databricks notebook.
- Inspect the Step 6 verification cells, especially the parser repair counts and amount/date coverage.
- Run `CreateAwards.ipynb` after QA.
- Flip the tracker row from Step 4 to Complete only after production verification.
